In [ ]:
# ========== 安装依赖：QLoRA 微调销售线索分级所需的库（版本钉死，避免接口漂移） ==========
# transformers：加载基座模型与分词器（Tokenizer）
!pip install -q transformers==4.47.0
# peft：Parameter-Efficient Fine-Tuning，提供 LoRA / QLoRA 适配器接口
!pip install -q peft==0.14.0
# trl：Transformer Reinforcement Learning 工具箱；这里主要用 SFTTrainer 做监督微调
!pip install -q trl==0.13.0
# bitsandbytes：4-bit 量化（Quantization），降低显存占用以便在消费级 GPU 上跑
!pip install -q bitsandbytes==0.45.0
# datasets：Hugging Face Dataset 对象，方便交给 Trainer
!pip install -q datasets==3.2.0
# accelerate：设备映射 / 分布式与混合精度辅助
!pip install -q accelerate==1.2.0


In [ ]:
# ========== 导入：把后面 QLoRA 流水线要用的工具箱搬进来 ==========

# 标准库 json：序列化/反序列化（本笔记本后续主要用 Dataset，这里预留）
import json
# 标准库 random：合成数据时随机抽样行业、预算等字段，并固定种子保证可复现
import random
# pandas：用 DataFrame 装线索表、做切分与预览
import pandas as pd
# Hugging Face Dataset：把 pandas 列包装成 Trainer 可吃的数据集
from datasets import Dataset
# transformers：分词器、因果语言模型、4-bit 配置、训练参数类
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
# peft：LoRA 配置、把基座包成可训练 PEFT 模型、任务类型枚举
from peft import LoraConfig, get_peft_model, TaskType
# trl：监督微调（Supervised Fine-Tuning）训练器
from trl import SFTTrainer
# torch：张量与设备；后面 generate / no_grad 会用到
import torch


In [ ]:
# ========== 常量与配置：模型路径、数据规模、LoRA/训练超参集中管理 ==========

# --- 模型 ---
# 基座 Instruct 模型 id（Hugging Face Hub 路径；需有权限/已登录）
BASE_MODEL      = "meta-llama/Llama-3.2-1B-Instruct"
# QLoRA 适配器与分词器输出目录（相对当前工作目录）
ADAPTER_OUT_DIR = "./qlora-sales-qualifier"

# --- 数据 ---
# 合成线索总条数（后面按三类大致均分）
NUM_SAMPLES     = 500
# 训练占比：0.85 → 约 425 train / 75 eval
TRAIN_SPLIT     = 0.85  # 425 train / 75 eval

# --- 训练（LoRA + SFT） ---
# LoRA 秩 r：低秩矩阵维度，越大表达能力越强、参数也越多
LORA_R          = 8
# LoRA alpha：缩放系数，常与 r 成比例（如 2*r）
LORA_ALPHA      = 16
# LoRA dropout：适配器路径上的随机丢弃，减轻过拟合
LORA_DROPOUT    = 0.05
# 学习率：QLoRA 常见量级 1e-4 ~ 2e-4
LEARNING_RATE   = 2e-4
# 训练轮数（epochs）
EPOCHS          = 3
# 每设备 batch size（再配合梯度累积放大有效 batch）
BATCH_SIZE      = 4
# 序列最大长度：超过会截断，影响显存与上下文覆盖
MAX_SEQ_LENGTH  = 512

# --- 可复现性（Reproducibility） ---
# 随机种子：固定后合成数据与训练抽样更可复现
SEED            = 42
# 给 Python random 设种子（生成线索时用）
random.seed(SEED)


In [ ]:
# ========== 合成 CRM 线索数据：用规则造 Hot/Warm/Cold，供指令微调 ==========

# 行业候选列表（字符串保持英文，便于模型学到课程同款字段）
INDUSTRIES    = ["SaaS", "FinTech", "Healthcare", "Retail", "Manufacturing", "Logistics", "EdTech"]
# 公司规模桶（员工数区间）
COMPANY_SIZES = ["1-10", "11-50", "51-200", "201-500", "501-1000", "1000+"]
# 线索来源渠道
LEAD_SOURCES  = ["Website", "LinkedIn", "Referral", "Cold Outreach", "Trade Show", "Webinar"]
# 痛点短语池：写入 has_pain_point 相关字段
PAIN_POINTS   = [
    "manual reporting", "poor lead tracking", "slow onboarding",
    "high churn", "lack of visibility", "disconnected tools", "scaling issues"
]

# 根据预算/规模/互动信号打分，再映射成 Hot / Warm / Cold 标签
def assign_label_and_score(lead: dict) -> tuple:
    # 累计分数从 0 起
    score = 0

    # 预算越高加分越多（销售资格的关键信号）
    if lead["budget_usd"] >= 50000:   score += 30
    elif lead["budget_usd"] >= 20000: score += 18
    else:                             score += 5

    # 公司规模：中大型加分更多
    if lead["company_size"] in ["201-500", "501-1000", "1000+"]: score += 20
    elif lead["company_size"] in ["51-200"]:                     score += 12
    else:                                                         score += 4

    # 浏览页数：每页 +2，封顶 10
    score += min(lead["pages_visited"] * 2, 10)
    # 打开邮件数：每封 +3，封顶 15
    score += min(lead["emails_opened"] * 3, 15)
    # 主动申请 demo：强意向信号
    if lead["requested_demo"]:   score += 15
    # 明确痛点：也加分
    if lead["has_pain_point"]:   score += 10

    # 分数夹在 0–100
    score = min(score, 100)

    # 阈值分档：>=65 Hot；>=35 Warm；否则 Cold
    if score >= 65:   label = "Hot"
    elif score >= 35: label = "Warm"
    else:             label = "Cold"

    # 返回标签与分数，供监督信号使用
    return label, score


# 生成偏「热线索」的原始字段分布（大预算、高互动、要 demo）
def generate_hot_lead() -> dict:
    return {
        "industry":       random.choice(INDUSTRIES),
        "company_size":   random.choice(["201-500", "501-1000", "1000+"]),
        "lead_source":    random.choice(LEAD_SOURCES),
        "budget_usd":     random.choice([50000, 75000, 100000]),
        "pages_visited":  random.randint(6, 10),
        "emails_opened":  random.randint(3, 5),
        "requested_demo": True,
        "has_pain_point": True,
        "pain_point":     random.choice(PAIN_POINTS),
    }

# 生成偏「温线索」的分布（中等预算/规模，互动中等）
def generate_warm_lead() -> dict:
    return {
        "industry":       random.choice(INDUSTRIES),
        "company_size":   random.choice(["51-200", "201-500"]),
        "lead_source":    random.choice(LEAD_SOURCES),
        "budget_usd":     random.choice([20000, 35000]),
        "pages_visited":  random.randint(3, 6),
        "emails_opened":  random.randint(1, 3),
        "requested_demo": random.choice([True, False]),
        "has_pain_point": random.choice([True, False]),
        "pain_point":     random.choice(PAIN_POINTS),
    }

# 生成偏「冷线索」的分布（小公司、低预算、低互动）
def generate_cold_lead() -> dict:
    return {
        "industry":       random.choice(INDUSTRIES),
        "company_size":   random.choice(["1-10", "11-50"]),
        "lead_source":    random.choice(LEAD_SOURCES),
        "budget_usd":     random.choice([5000, 10000]),
        "pages_visited":  random.randint(1, 3),
        "emails_opened":  random.randint(0, 1),
        "requested_demo": False,
        "has_pain_point": False,
        "pain_point":     random.choice(PAIN_POINTS),
    }


# --- 生成近似平衡的数据集（每类约 NUM_SAMPLES//3） ---
# 每类条数（整数除法；总数可能略少于 NUM_SAMPLES）
per_class = NUM_SAMPLES // 3
# 汇总列表
leads = []
# 依次用三个生成器各造一批
for fn in [generate_hot_lead, generate_warm_lead, generate_cold_lead]:
    # 列表推导：调用生成器 per_class 次
    batch = [fn() for _ in range(per_class)]
    # 对每条线索用规则函数补 label / score（监督标签）
    for lead in batch:
        lead["label"], lead["score"] = assign_label_and_score(lead)
    # 拼进总列表
    leads.extend(batch)

# 打乱顺序，避免按类成块喂给训练
random.shuffle(leads)
# 转成 DataFrame，方便切分与 apply 格式化
df = pd.DataFrame(leads)

# 看标签分布是否大致均衡
print(df["label"].value_counts())
# 预览前 3 行字段
print(df.head(3))


In [ ]:
# ========== 提示格式化：把每条线索变成「指令微调」用的 chat 文本对 ==========

# System 提示：规定助手角色与严格输出格式（可运行字符串，保持英文原文）
SYSTEM_PROMPT = (
    "You are a sales qualification assistant. "
    "Given CRM data about an inbound lead, return a qualification label (Hot, Warm, or Cold) "
    "and a score from 0 to 100. Respond in this exact format:\n"
    "Label: <Hot|Warm|Cold>\nScore: <0-100>\nReason: <one sentence>"
)

# 把一条 lead dict 拼成 system/user/assistant 三段式训练文本
def format_prompt(lead: dict) -> str:
    # 用户侧：CRM 字段逐行列出（与推理时输入格式对齐）
    user_msg = (
        f"Industry: {lead['industry']}\n"
        f"Company Size: {lead['company_size']} employees\n"
        f"Lead Source: {lead['lead_source']}\n"
        f"Budget: ${lead['budget_usd']:,}\n"
        f"Pages Visited: {lead['pages_visited']}\n"
        f"Emails Opened: {lead['emails_opened']}\n"
        f"Requested Demo: {lead['requested_demo']}\n"
        f"Has Pain Point: {lead['has_pain_point']} ({lead['pain_point']})"
    )
    # 助手侧：标签 + 分数 + 一句理由（监督目标）
    assistant_msg = (
        f"Label: {lead['label']}\n"
        f"Score: {lead['score']}\n"
        f"Reason: This lead scores {lead['score']}/100 based on budget, engagement, and fit signals."
    )
    # 用简单特殊标记包三段（与本笔记本推理侧一致）
    return (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{user_msg}\n"
        f"<|assistant|>\n{assistant_msg}"
    )


# 对每一行 lead 生成 text 列（SFT 只吃这一列）
df["text"] = df.apply(format_prompt, axis=1)

# --- 训练 / 评估切分 ---
# 按 TRAIN_SPLIT 算切分下标（前段 train，后段 eval）
split_idx = int(len(df) * TRAIN_SPLIT)
# 只保留 text 列，转成 HF Dataset
train_dataset = Dataset.from_pandas(df[["text"]].iloc[:split_idx].reset_index(drop=True))
eval_dataset  = Dataset.from_pandas(df[["text"]].iloc[split_idx:].reset_index(drop=True))

# 打印样本数，确认切分
print(f"Train samples : {len(train_dataset)}")
print(f"Eval samples  : {len(eval_dataset)}")
print("\nSample prompt:\n")
# 看一条完整训练文本长什么样
print(df["text"].iloc[0])


In [ ]:
# ========== 登录 Hugging Face Hub：拉取门控模型（如 Llama）前需要凭证 ==========
# login()：交互式或环境里已有 token 时完成鉴权
from huggingface_hub import login; login()


In [ ]:
# ========== 加载基座模型 + 分词器（4-bit NF4 量化，省显存） ==========

# BitsAndBytes 4-bit 配置：QLoRA 的「Q」部分
bnb_config = BitsAndBytesConfig(
    # 以 4-bit 权重加载
    load_in_4bit=True,
    # 双重量化：再压一点量化常数的开销
    bnb_4bit_use_double_quant=True,
    # 量化类型 NF4（NormalFloat4），对 LLM 权重常见更稳
    bnb_4bit_quant_type="nf4",
    # 计算时用 float16（与 fp16 训练常见搭配）
    bnb_4bit_compute_dtype=torch.float16,
)

# 从 Hub 拉与 BASE_MODEL 匹配的分词器
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
# Causal LM 常缺 pad_token：用 eos 顶上，避免 batch padding 报错
tokenizer.pad_token = tokenizer.eos_token
# 右侧 padding：生成任务更常见（左侧会干扰因果注意力位置）
tokenizer.padding_side = "right"

# 按 4-bit 配置加载因果语言模型；device_map=auto 自动铺到可用 GPU
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
# 训练时关掉 KV cache（与梯度检查点等训练设置更合拍）
model.config.use_cache = False

# 确认加载成功与设备映射
print(f"Model loaded: {BASE_MODEL}")
print(f"Device map  : {model.hf_device_map}")


In [ ]:
# ========== LoRA 配置：只训练低秩适配器，冻结绝大部分基座权重 ==========

# 构造 LoRAConfig：秩、缩放、dropout、作用模块
lora_config = LoraConfig(
    # 低秩维度 r
    r=LORA_R,
    # 缩放 alpha
    lora_alpha=LORA_ALPHA,
    # 适配器 dropout
    lora_dropout=LORA_DROPOUT,
    # 不训练 bias
    bias="none",
    # 因果语言建模任务
    task_type=TaskType.CAUSAL_LM,
    # 注入注意力投影：q/k/v/o（Llama 常见目标模块名）
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# 把基座包成 PEFT 模型（可训练参数只剩 LoRA）
model = get_peft_model(model, lora_config)
# 打印可训练参数占比，确认 QLoRA 生效
model.print_trainable_parameters()


In [ ]:
# ========== 用 SFTTrainer / SFTConfig 做监督微调 ==========

# SFTConfig：trl 侧训练超参（含 max_seq_length 等 SFT 专用字段）
from trl import SFTConfig

# 组装训练参数：输出目录、轮数、batch、学习率、评估与保存策略
training_args = SFTConfig(
    # checkpoint / 适配器落盘目录
    output_dir=ADAPTER_OUT_DIR,
    # 训练轮数
    num_train_epochs=EPOCHS,
    # 每设备训练 batch
    per_device_train_batch_size=BATCH_SIZE,
    # 每设备评估 batch
    per_device_eval_batch_size=BATCH_SIZE,
    # 梯度累积：有效 batch ≈ BATCH_SIZE * 4
    gradient_accumulation_steps=4,
    # 学习率
    learning_rate=LEARNING_RATE,
    # fp16 混合精度，省显存、加速
    fp16=True,
    # 每 10 step 打一次 log
    logging_steps=10,
    # 每个 epoch 评估一次
    eval_strategy="epoch",
    # 每个 epoch 存一次
    save_strategy="epoch",
    # 结束时加载验证集上最好的 checkpoint
    load_best_model_at_end=True,
    # 序列最大长度
    max_seq_length=MAX_SEQ_LENGTH,
    # 训练随机种子
    seed=SEED,
)

# 组装 SFTTrainer：模型 + 分词器(processing_class) + 数据
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# 启动训练（耗时取决于 GPU；需已装好 CUDA / bitsandbytes）
trainer.train()


In [ ]:
# ========== 保存 LoRA 适配器与分词器到 ADAPTER_OUT_DIR ==========

# 只保存 PEFT 适配器权重（体积远小于整模）
model.save_pretrained(ADAPTER_OUT_DIR)
# 同步保存分词器配置，推理时成套加载
tokenizer.save_pretrained(ADAPTER_OUT_DIR)

# 打印落盘路径
print(f"Adapter saved to: {ADAPTER_OUT_DIR}")
# shell：列出输出目录内容，确认 adapter_config / tokenizer 等文件在
!ls {ADAPTER_OUT_DIR}


In [ ]:
# ========== 推理与抽查：加载适配器，对几条测试线索做资格分级 ==========

# PeftModel：把已保存的 LoRA 挂到基座上
from peft import PeftModel

# 重新以 4-bit 加载干净基座（与训练时量化设置对齐）
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
    device_map="auto",
)
# 从目录挂上微调适配器
fine_tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_OUT_DIR)
# 评估模式：关掉 dropout 等训练行为
fine_tuned_model.eval()

# 对单条 lead 做生成式资格预测，并抽出 Label/Score/Reason 三行
def qualify_lead(lead: dict) -> str:
    # 构造与训练一致的 system/user 前缀，以 assistant 标记结尾让模型续写
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n"
        f"Industry: {lead['industry']}\n"
        f"Company Size: {lead['company_size']} employees\n"
        f"Lead Source: {lead['lead_source']}\n"
        f"Budget: ${lead['budget_usd']:,}\n"
        f"Pages Visited: {lead['pages_visited']}\n"
        f"Emails Opened: {lead['emails_opened']}\n"
        f"Requested Demo: {lead['requested_demo']}\n"
        f"Has Pain Point: {lead['has_pain_point']} ({lead['pain_point']})\n"
        f"<|assistant|>\n"
    )
    # 分词并搬到模型所在设备
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)
    # 推理不需要梯度，省显存
    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            # 最多新生成 80 token
            max_new_tokens=80,
            # 贪婪解码，结果更稳定便于对比
            do_sample=False,
            # 重复惩罚，减轻啰嗦循环
            repetition_penalty=1.3,
            # 遇到 eos 停止
            eos_token_id=tokenizer.eos_token_id,
        )
    # 整段解码（含 prompt）
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 只保留 assistant 段，并过滤出 Label/Score/Reason 行
    assistant_response = response.split("<|assistant|>")[-1].strip()
    lines = [l for l in assistant_response.splitlines() if l.startswith(("Label:", "Score:", "Reason:"))]
    # 最多返回三行，避免多余续写
    return "\n".join(lines[:3])


# --- 三条手工测试线索：偏热 / 偏冷 / 偏温 ---
test_leads = [
    {"industry": "SaaS", "company_size": "201-500", "lead_source": "Referral",
     "budget_usd": 75000, "pages_visited": 8, "emails_opened": 4,
     "requested_demo": True, "has_pain_point": True, "pain_point": "high churn"},

    {"industry": "Retail", "company_size": "1-10", "lead_source": "Cold Outreach",
     "budget_usd": 5000, "pages_visited": 2, "emails_opened": 0,
     "requested_demo": False, "has_pain_point": False, "pain_point": "manual reporting"},

    {"industry": "FinTech", "company_size": "51-200", "lead_source": "Webinar",
     "budget_usd": 20000, "pages_visited": 5, "emails_opened": 2,
     "requested_demo": False, "has_pain_point": True, "pain_point": "disconnected tools"},
]

# 逐条打印模型输出
for i, lead in enumerate(test_leads, 1):
    print(f"--- Lead {i} ---")
    print(qualify_lead(lead))
    print()
